# Phase 1–2: leakage-safe target construction and Logistic V1

- One row represents one video chunk send event.
- The authoritative freeze start is the official client event `event == "rebuffer"`.
- `target_5s = 1` when a freeze begins in the same `session_id + index` strictly after the sent row and within the next five seconds.
- Rows already inside a freeze are excluded because the goal is early prediction.
- Startup and unknown playback state are excluded.
- Future ACK information is excluded because it is unavailable at prediction time.
- Full interpretation is recorded in `docs/Phase1_Research_Flow_CN.md` and `docs/Phase2_Modelling_Report_CN.md`.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent

REAL_RAW = ROOT / "data" / "raw" / "real" / "2026-07-19"

sent_path = next(REAL_RAW.glob("video_sent_*.csv"))

print("Project root:", ROOT)
print("Video sent file:", sent_path)

Project root: /Users/oliver/Downloads/video_rebuffering_risk_lab_fixed
Video sent file: /Users/oliver/Downloads/video_rebuffering_risk_lab_fixed/data/raw/real/2026-07-19/video_sent_2026-07-19T11_2026-07-20T11.csv


In [2]:
sent_sample = pd.read_csv(
    sent_path,
    nrows=1000
)

print("Shape:", sent_sample.shape)
print("\nColumns:")
print(sent_sample.columns.tolist())

print("\nData types:")
print(sent_sample.dtypes)

display(sent_sample.head())

Shape: (1000, 16)

Columns:
['time (ns GMT)', 'session_id', 'index', 'expt_id', 'channel', 'video_ts', 'format', 'size', 'ssim_index', 'cwnd', 'in_flight', 'min_rtt', 'rtt', 'delivery_rate', 'buffer', 'cum_rebuf']

Data types:
time (ns GMT)      int64
session_id        object
index              int64
expt_id            int64
channel           object
video_ts           int64
format            object
size               int64
ssim_index       float64
cwnd               int64
in_flight          int64
min_rtt            int64
rtt                int64
delivery_rate      int64
buffer           float64
cum_rebuf        float64
dtype: object


,time (ns GMT),session_id,index,expt_id,channel,video_ts,format,size,ssim_index,cwnd,in_flight,min_rtt,rtt,delivery_rate,buffer,cum_rebuf
0,1784461186794000000,eMS9pWUdc8E/0bTtn/MWzYRSEwojcjZ8DMu+RveNx0g=,2,2240,abc,16423767360,1280x720-20,780344,0.974398,1674,0,67043,98722,6175404,0.000,0.000
1,1784461187130000000,eMS9pWUdc8E/0bTtn/MWzYRSEwojcjZ8DMu+RveNx0g=,2,2240,abc,16423947540,1280x720-20,673278,0.975522,612,0,67043,102152,54984,2.002,0.000
2,1784461187457000000,eMS9pWUdc8E/0bTtn/MWzYRSEwojcjZ8DMu+RveNx0g=,2,2240,abc,16424127720,1280x720-20,742545,0.975965,602,0,67043,74561,3417132,3.789,0.346
3,1784461187787000000,eMS9pWUdc8E/0bTtn/MWzYRSEwojcjZ8DMu+RveNx0g=,2,2240,abc,16424307900,1280x720-20,912239,0.975388,477,0,67043,77988,2586248,5.440,0.346
4,1784461188203000000,eMS9pWUdc8E/0bTtn/MWzYRSEwojcjZ8DMu+RveNx0g=,2,2240,abc,16424488080,1280x720-20,774798,0.973415,554,0,67043,88670,3247811,7.040,0.346


In [3]:
client_path = next(
    REAL_RAW.glob("client_buffer_*.csv")
)

print(client_path)

/Users/oliver/Downloads/video_rebuffering_risk_lab_fixed/data/raw/real/2026-07-19/client_buffer_2026-07-19T11_2026-07-20T11.csv


In [4]:
buffer_sample = pd.read_csv(
    client_path,
    nrows=500_000,
    usecols=[
        "time (ns GMT)",
        "session_id",
        "index",
        "event",
        "buffer",
        "cum_rebuf"
    ]
)

### Decision: reject the `cum_rebuf`-increment freeze-start rule

The timeline above is the key audit. The official `rebuffer` row occurs when buffer is nearly exhausted, while `cum_rebuf` increases at the later `play` row when playback resumes. Therefore the earlier cells that used `cum_rebuf_change > 0` are retained only as a rejected exploratory approach; their `freeze_start` output must not be used downstream.

Final authoritative rule:

```python
freeze_start = client_buffer["event"] == "rebuffer"
```

This correction prevents a freeze end/resume update from being mislabeled as the freeze start.


In [5]:
buffer_sample = buffer_sample.sort_values(
    ["session_id", "index", "time (ns GMT)"]
).copy()

In [6]:
buffer_sample["previous_cum_rebuf"] = (
    buffer_sample
    .groupby(["session_id", "index"])["cum_rebuf"]
    .shift(1)
)

In [7]:
buffer_sample["cum_rebuf_change"] = (
    buffer_sample["cum_rebuf"]
    - buffer_sample["previous_cum_rebuf"]
)

In [8]:
buffer_sample["is_rebuffering"] = (
    (buffer_sample["cum_rebuf_change"] > 0)
    & (buffer_sample["event"] != "startup")
)

In [9]:
buffer_sample["previous_is_rebuffering"] = (
    buffer_sample
    .groupby(["session_id", "index"])["is_rebuffering"]
    .shift(1, fill_value=False)
)

In [10]:
buffer_sample["freeze_start"] = (
    buffer_sample["is_rebuffering"]
    & ~buffer_sample["previous_is_rebuffering"]
)

In [11]:
freeze_starts_sample = buffer_sample[
    buffer_sample["freeze_start"]
].copy()

print("Freeze starts found:", len(freeze_starts_sample))

display(
    freeze_starts_sample[
        [
            "time (ns GMT)",
            "session_id",
            "index",
            "event",
            "buffer",
            "previous_cum_rebuf",
            "cum_rebuf",
            "cum_rebuf_change"
        ]
    ].head(20)
)

Freeze starts found: 16


,time (ns GMT),session_id,index,event,buffer,previous_cum_rebuf,cum_rebuf,cum_rebuf_change
412607,1784472128562000000,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,timer,0.018,0.170,0.217,0.047
412625,1784472130551000000,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,timer,0.025,0.418,0.473,0.055
460734,1784475678288000000,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,play,1.862,1.317,1.465,0.148
477016,1784476358840000000,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,timer,0.000,1.465,1.558,0.093
480429,1784476501092000000,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,timer,0.000,2.713,2.914,0.201
480755,1784476514598000000,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,timer,0.462,15.667,15.812,0.145
480787,1784476515850000000,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,timer,0.000,16.412,16.652,0.240
484658,1784476677848000000,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,timer,0.000,17.403,17.491,0.088
485270,1784476706342000000,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,timer,0.000,24.341,24.583,0.242
486078,1784476746346000000,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,timer,0.000,24.583,24.784,0.201


In [12]:
selected_session = "Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8="
selected_index = 8
center_time = 1784475678288000000

two_seconds_ns = 2_000_000_000

timeline = buffer_sample[
    (buffer_sample["session_id"] == selected_session)
    & (buffer_sample["index"] == selected_index)
    & (
        buffer_sample["time (ns GMT)"]
        .between(
            center_time - two_seconds_ns,
            center_time + two_seconds_ns
        )
    )
].copy()

display(
    timeline[
        [
            "time (ns GMT)",
            "event",
            "buffer",
            "cum_rebuf",
            "cum_rebuf_change",
            "is_rebuffering",
            "freeze_start"
        ]
    ]
)

,time (ns GMT),event,buffer,cum_rebuf,cum_rebuf_change,is_rebuffering,freeze_start
460687,1784475676350000000,timer,1.799,1.317,0.000,False,False
460693,1784475676596000000,timer,1.550,1.317,0.000,False,False
460698,1784475676836000000,timer,1.314,1.317,0.000,False,False
460705,1784475677091000000,timer,1.053,1.317,0.000,False,False
460711,1784475677340000000,timer,0.806,1.317,0.000,False,False
460717,1784475677584000000,timer,0.561,1.317,0.000,False,False
460723,1784475677835000000,timer,0.309,1.317,0.000,False,False
460729,1784475678088000000,rebuffer,0.060,1.317,0.000,False,False
460730,1784475678088000001,timer,0.059,1.317,0.000,False,False
460734,1784475678288000000,play,1.862,1.465,0.148,True,True


In [13]:
freeze_starts_sample = (
    buffer_sample[
        buffer_sample["event"] == "rebuffer"
    ][
        [
            "session_id",
            "index",
            "time (ns GMT)",
            "buffer",
            "cum_rebuf"
        ]
    ]
    .rename(
        columns={
            "time (ns GMT)": "freeze_start_time"
        }
    )
    .copy()
)

In [14]:
print(
    "Freeze starts found:",
    len(freeze_starts_sample)
)

display(freeze_starts_sample.head(20))

Freeze starts found: 19


,session_id,index,freeze_start_time,buffer,cum_rebuf
264938,EJrlAQLTEDfUYS3HDszx+aKRAaqtv6Eo1D95xkNl4qI=,0,1784512705089000000,0.058,2.251
412605,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,1784472128506000000,0.064,0.170
412623,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,1784472130494000000,0.079,0.418
423998,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,1784473503858000000,0.099,1.317
460729,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,1784475678088000000,0.060,1.317
477013,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,1784476358746000000,0.083,1.465
480423,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,1784476500898000000,0.081,2.713
480751,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,1784476514450000000,0.507,15.667
480779,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,1784476515604000000,0.072,16.412
484655,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,1784476677758000000,0.085,17.403


In [15]:
freeze_parts = []

for chunk_number, chunk in enumerate(
    pd.read_csv(
        client_path,
        usecols=[
            "time (ns GMT)",
            "session_id",
            "index",
            "event",
            "buffer",
            "cum_rebuf"
        ],
        chunksize=500_000
    ),
    start=1
):
    found = chunk[
        chunk["event"] == "rebuffer"
    ][
        [
            "session_id",
            "index",
            "time (ns GMT)",
            "buffer",
            "cum_rebuf"
        ]
    ].copy()

    freeze_parts.append(found)

    print(
        f"Chunk {chunk_number}: "
        f"{len(found)} freeze starts found"
    )

Chunk 1: 19 freeze starts found
Chunk 2: 8 freeze starts found
Chunk 3: 16 freeze starts found
Chunk 4: 35 freeze starts found
Chunk 5: 5 freeze starts found
Chunk 6: 123 freeze starts found
Chunk 7: 223 freeze starts found
Chunk 8: 74 freeze starts found
Chunk 9: 771 freeze starts found
Chunk 10: 50 freeze starts found
Chunk 11: 20 freeze starts found
Chunk 12: 13 freeze starts found
Chunk 13: 4 freeze starts found
Chunk 14: 1 freeze starts found
Chunk 15: 149 freeze starts found
Chunk 16: 53 freeze starts found
Chunk 17: 5 freeze starts found
Chunk 18: 9 freeze starts found
Chunk 19: 8 freeze starts found
Chunk 20: 24 freeze starts found
Chunk 21: 0 freeze starts found
Chunk 22: 2 freeze starts found
Chunk 23: 16 freeze starts found
Chunk 24: 3 freeze starts found
Chunk 25: 6 freeze starts found
Chunk 26: 15 freeze starts found
Chunk 27: 12 freeze starts found
Chunk 28: 0 freeze starts found
Chunk 29: 3 freeze starts found
Chunk 30: 127 freeze starts found
Chunk 31: 5 freeze starts f

In [16]:
freeze_starts = pd.concat(
    freeze_parts,
    ignore_index=True
)

freeze_starts = freeze_starts.rename(
    columns={
        "time (ns GMT)": "freeze_start_time"
    }
)

In [17]:
print("Total freeze starts:", len(freeze_starts))

print(
    "Streams with freeze:",
    freeze_starts[
        ["session_id", "index"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate freeze keys:",
    freeze_starts.duplicated(
        ["session_id", "index", "freeze_start_time"]
    ).sum()
)

display(freeze_starts.head())

display(
    freeze_starts["buffer"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)

Total freeze starts: 5935
Streams with freeze: 671
Duplicate freeze keys: 0


,session_id,index,freeze_start_time,buffer,cum_rebuf
0,oSx44INgCDpIsXdR1rwH4iy1OsaLeYR9wr+0jIAtTPg=,9,1784521331179000000,0.000,0.060
1,EJrlAQLTEDfUYS3HDszx+aKRAaqtv6Eo1D95xkNl4qI=,0,1784512705089000000,0.058,2.251
2,smLdptbJIYKyIHE3SBFiP+bUMl9CiCHXXd7ARacP9Kg=,0,1784515779273000000,0.060,1.353
3,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,1784472128506000000,0.064,0.170
4,Ftd9ukvSITLlbrxmxgismmxL9IxOzeKL2jXJ72w5qA8=,8,1784472130494000000,0.079,0.418


count    5935.000000
mean        0.200105
std         0.397393
min         0.000000
50%         0.078000
90%         0.573600
95%         1.280900
99%         1.871660
max         3.826000
Name: buffer, dtype: float64

In [18]:
freeze_lookup = freeze_starts[
    [
        "session_id",
        "index",
        "freeze_start_time"
    ]
].copy()

freeze_lookup = freeze_lookup.sort_values(
    [
        "freeze_start_time",
        "session_id",
        "index"
    ]
)

In [19]:
sent_chunk = pd.read_csv(
    sent_path,
    nrows=500_000
)

sent_chunk = sent_chunk.rename(
    columns={
        "time (ns GMT)": "sent_time"
    }
)

sent_chunk = sent_chunk.sort_values(
    [
        "sent_time",
        "session_id",
        "index"
    ]
)

In [20]:
five_seconds_ns = 5_000_000_000

labelled_chunk = pd.merge_asof(
    sent_chunk,
    freeze_lookup,
    left_on="sent_time",
    right_on="freeze_start_time",
    by=["session_id", "index"],
    direction="forward",
    tolerance=five_seconds_ns
)

In [21]:
labelled_chunk["target_5s"] = (
    labelled_chunk["freeze_start_time"].notna()
    & (
        labelled_chunk["freeze_start_time"]
        > labelled_chunk["sent_time"]
    )
).astype(int)

In [22]:
print(
    labelled_chunk["target_5s"].value_counts()
)

display(
    labelled_chunk[
        labelled_chunk["target_5s"] == 1
    ][
        [
            "session_id",
            "index",
            "sent_time",
            "freeze_start_time",
            "buffer",
            "rtt",
            "delivery_rate",
            "target_5s"
        ]
    ].head(20)
)

target_5s
0    499286
1       714
Name: count, dtype: int64


,session_id,index,sent_time,freeze_start_time,buffer,rtt,delivery_rate,target_5s
536,raX170SYGeEp5HuJvod8o7mNXDglg7vkiX8nWw7esVs=,0,1784459032681000000,1.784459e+18,2.002,100762,16240,1
1066,raX170SYGeEp5HuJvod8o7mNXDglg7vkiX8nWw7esVs=,1,1784459199283000000,1.784459e+18,0.465,125971,54729,1
1079,raX170SYGeEp5HuJvod8o7mNXDglg7vkiX8nWw7esVs=,1,1784459204103000000,1.784459e+18,0.943,236053,44285,1
2138,raX170SYGeEp5HuJvod8o7mNXDglg7vkiX8nWw7esVs=,1,1784459527161000000,1.784460e+18,4.408,186466,27836,1
2178,raX170SYGeEp5HuJvod8o7mNXDglg7vkiX8nWw7esVs=,1,1784459540285000000,1.784460e+18,1.542,170042,24133,1
2187,raX170SYGeEp5HuJvod8o7mNXDglg7vkiX8nWw7esVs=,1,1784459543104000000,1.784460e+18,1.339,211139,21493,1
2192,raX170SYGeEp5HuJvod8o7mNXDglg7vkiX8nWw7esVs=,1,1784459544348000000,1.784460e+18,2.101,207327,42201,1
23043,raX170SYGeEp5HuJvod8o7mNXDglg7vkiX8nWw7esVs=,1,1784465048192000000,1.784465e+18,0.000,83279,18043,1
23049,raX170SYGeEp5HuJvod8o7mNXDglg7vkiX8nWw7esVs=,1,1784465049205000000,1.784465e+18,1.047,79362,3826,1
23084,raX170SYGeEp5HuJvod8o7mNXDglg7vkiX8nWw7esVs=,1,1784465057474000000,1.784465e+18,3.105,94052,26120,1


In [23]:
positive_chunk = labelled_chunk[
    labelled_chunk["target_5s"] == 1
].copy()

positive_chunk["seconds_until_freeze"] = (
    positive_chunk["freeze_start_time"]
    - positive_chunk["sent_time"]
) / 1_000_000_000

In [24]:
display(
    positive_chunk["seconds_until_freeze"]
    .describe(
        percentiles=[0.01, 0.1, 0.5, 0.9, 0.99]
    )
)

count    714.000000
mean       2.142887
std        1.562217
min        0.001000
1%         0.021780
10%        0.191300
50%        2.064000
90%        4.480400
99%        4.947870
max        4.986000
Name: seconds_until_freeze, dtype: float64

In [25]:
print(
    "Not after sent:",
    (
        positive_chunk["seconds_until_freeze"] <= 0
    ).sum()
)

print(
    "Beyond 5 seconds:",
    (
        positive_chunk["seconds_until_freeze"] > 5
    ).sum()
)

Not after sent: 0
Beyond 5 seconds: 0


In [26]:
state_parts = []

state_event_names = [
    "init",
    "startup",
    "rebuffer",
    "play"
]

for chunk_number, chunk in enumerate(
    pd.read_csv(
        client_path,
        usecols=[
            "time (ns GMT)",
            "session_id",
            "index",
            "event"
        ],
        chunksize=500_000
    ),
    start=1
):
    found = chunk[
        chunk["event"].isin(state_event_names)
    ].copy()

    state_parts.append(found)

    print(
        f"Chunk {chunk_number}: "
        f"{len(found)} state events"
    )

Chunk 1: 535 state events
Chunk 2: 82 state events
Chunk 3: 637 state events
Chunk 4: 545 state events
Chunk 5: 364 state events
Chunk 6: 324 state events
Chunk 7: 878 state events
Chunk 8: 604 state events
Chunk 9: 1832 state events
Chunk 10: 123 state events
Chunk 11: 878 state events
Chunk 12: 542 state events
Chunk 13: 66 state events
Chunk 14: 104 state events
Chunk 15: 644 state events
Chunk 16: 604 state events
Chunk 17: 259 state events
Chunk 18: 359 state events
Chunk 19: 260 state events
Chunk 20: 448 state events
Chunk 21: 50 state events
Chunk 22: 56 state events
Chunk 23: 983 state events
Chunk 24: 57 state events
Chunk 25: 79 state events
Chunk 26: 738 state events
Chunk 27: 430 state events
Chunk 28: 41 state events
Chunk 29: 747 state events
Chunk 30: 597 state events
Chunk 31: 40 state events
Chunk 32: 73 state events
Chunk 33: 286 state events
Chunk 34: 446 state events
Chunk 35: 51 state events
Chunk 36: 77 state events
Chunk 37: 646 state events
Chunk 38: 240 state 

In [27]:
state_events = pd.concat(
    state_parts,
    ignore_index=True
)

state_events = state_events.rename(
    columns={
        "time (ns GMT)": "client_event_time"
    }
)

In [28]:
state_map = {
    "init": "startup_buffering",
    "startup": "playing",
    "rebuffer": "rebuffering",
    "play": "playing"
}

state_events["playback_state"] = (
    state_events["event"].map(state_map)
)

In [29]:
state_events = state_events.sort_values(
    [
        "client_event_time",
        "session_id",
        "index"
    ]
)

state_output = (
    ROOT
    / "data"
    / "interim"
    / "playback_state_events.parquet"
)

state_output.parent.mkdir(
    parents=True,
    exist_ok=True
)

state_events.to_parquet(
    state_output,
    index=False
)

In [30]:
print("Total state events:", len(state_events))

print("\nEvent counts:")
print(state_events["event"].value_counts())

print("\nPlayback states:")
print(state_events["playback_state"].value_counts())

print("\nSaved to:")
print(state_output)

Total state events: 80407

Event counts:
event
init        34766
startup     34106
rebuffer     5935
play         5600
Name: count, dtype: int64

Playback states:
playback_state
playing              39706
startup_buffering    34766
rebuffering           5935
Name: count, dtype: int64

Saved to:
/Users/oliver/Downloads/video_rebuffering_risk_lab_fixed/data/interim/playback_state_events.parquet


In [31]:
state_lookup = state_events[
    [
        "session_id",
        "index",
        "client_event_time",
        "event",
        "playback_state"
    ]
].rename(
    columns={
        "event": "last_client_event"
    }
).sort_values(
    [
        "client_event_time",
        "session_id",
        "index"
    ]
)

freeze_lookup = freeze_starts[
    [
        "session_id",
        "index",
        "freeze_start_time"
    ]
].sort_values(
    [
        "freeze_start_time",
        "session_id",
        "index"
    ]
)

In [32]:
from pathlib import Path
import shutil

output_dir = (
    ROOT
    / "data"
    / "processed"
    / "labelled_sent_5s"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

existing_parts = list(
    output_dir.glob("part_*.parquet")
)

if existing_parts:
    raise RuntimeError(
        f"Output directory already contains "
        f"{len(existing_parts)} parts: {output_dir}"
    )

free_gb = (
    shutil.disk_usage(ROOT).free
    / 1024**3
)

print(f"Free disk space: {free_gb:.2f} GB")
print("Output directory:", output_dir)

RuntimeError: Output directory already contains 24 parts: /Users/oliver/Downloads/video_rebuffering_risk_lab_fixed/data/processed/labelled_sent_5s

In [33]:
five_seconds_ns = 5_000_000_000

total_rows = 0
total_eligible = 0
total_positive = 0
total_rebuffering = 0
total_unknown_state = 0

for chunk_number, sent_chunk in enumerate(
    pd.read_csv(
        sent_path,
        chunksize=500_000
    ),
    start=1
):
    # 明确发送时间
    sent_chunk = sent_chunk.rename(
        columns={
            "time (ns GMT)": "sent_time"
        }
    )

    sent_chunk = sent_chunk.sort_values(
        [
            "sent_time",
            "session_id",
            "index"
        ]
    )

    # 向过去找最近的播放器状态事件
    labelled_chunk = pd.merge_asof(
        sent_chunk,
        state_lookup,
        left_on="sent_time",
        right_on="client_event_time",
        by=["session_id", "index"],
        direction="backward"
    )

    labelled_chunk["playback_state"] = (
        labelled_chunk["playback_state"]
        .fillna("unknown")
    )

    labelled_chunk["eligible_prediction"] = (
        labelled_chunk["playback_state"]
        == "playing"
    )

    # 向未来找最近的 freeze start
    labelled_chunk = pd.merge_asof(
        labelled_chunk.sort_values(
            [
                "sent_time",
                "session_id",
                "index"
            ]
        ),
        freeze_lookup,
        left_on="sent_time",
        right_on="freeze_start_time",
        by=["session_id", "index"],
        direction="forward",
        tolerance=five_seconds_ns
    )

    labelled_chunk["seconds_until_freeze"] = (
        labelled_chunk["freeze_start_time"]
        - labelled_chunk["sent_time"]
    ) / 1_000_000_000

    labelled_chunk["target_5s"] = (
        labelled_chunk["eligible_prediction"]
        & labelled_chunk["freeze_start_time"].notna()
        & (
            labelled_chunk["seconds_until_freeze"] > 0
        )
        & (
            labelled_chunk["seconds_until_freeze"] <= 5
        )
    ).astype("int8")

    # 保存这一批
    part_path = (
        output_dir
        / f"part_{chunk_number:03d}.parquet"
    )

    labelled_chunk.to_parquet(
        part_path,
        index=False,
        compression="snappy"
    )

    # 更新统计
    total_rows += len(labelled_chunk)
    total_eligible += int(
        labelled_chunk[
            "eligible_prediction"
        ].sum()
    )
    total_positive += int(
        labelled_chunk["target_5s"].sum()
    )
    total_rebuffering += int(
        (
            labelled_chunk["playback_state"]
            == "rebuffering"
        ).sum()
    )
    total_unknown_state += int(
        (
            labelled_chunk["playback_state"]
            == "unknown"
        ).sum()
    )

    print(
        f"Part {chunk_number:03d} | "
        f"rows={len(labelled_chunk):,} | "
        f"eligible={labelled_chunk['eligible_prediction'].sum():,} | "
        f"positive={labelled_chunk['target_5s'].sum():,}"
    )

Part 001 | rows=500,000 | eligible=464,031 | positive=579
Part 002 | rows=500,000 | eligible=438,042 | positive=374
Part 003 | rows=500,000 | eligible=449,841 | positive=33
Part 004 | rows=500,000 | eligible=439,132 | positive=65
Part 005 | rows=500,000 | eligible=366,524 | positive=11


KeyboardInterrupt: 

In [34]:
output_size_gb = sum(
    path.stat().st_size
    for path in output_dir.glob("part_*.parquet")
) / 1024**3

print("Total rows:", total_rows)
print("Eligible prediction rows:", total_eligible)
print("Positive target rows:", total_positive)
print("Rows currently rebuffering:", total_rebuffering)
print("Rows with unknown state:", total_unknown_state)
print(f"Parquet size: {output_size_gb:.2f} GB")

Total rows: 2500000
Eligible prediction rows: 2157570
Positive target rows: 1062
Rows currently rebuffering: 1383
Rows with unknown state: 330373
Parquet size: 0.36 GB


In [35]:
session_summary_parts = []

parquet_files = sorted(
    output_dir.glob("part_*.parquet")
)

print("Parquet files:", len(parquet_files))

Parquet files: 24


In [36]:
for file_number, parquet_file in enumerate(
    parquet_files,
    start=1
):
    part = pd.read_parquet(
        parquet_file,
        columns=[
            "session_id",
            "eligible_prediction",
            "target_5s"
        ]
    )

    # 只有 eligible rows 才进入模型
    part = part[
        part["eligible_prediction"]
    ]

    part_summary = (
        part
        .groupby("session_id")
        .agg(
            rows=("target_5s", "size"),
            positive_rows=("target_5s", "sum")
        )
        .reset_index()
    )

    session_summary_parts.append(
        part_summary
    )

    print(
        f"Finished file {file_number}: "
        f"{len(part_summary)} sessions"
    )

Finished file 1: 156 sessions
Finished file 2: 158 sessions
Finished file 3: 233 sessions
Finished file 4: 198 sessions
Finished file 5: 183 sessions
Finished file 6: 175 sessions
Finished file 7: 178 sessions
Finished file 8: 162 sessions
Finished file 9: 104 sessions
Finished file 10: 130 sessions
Finished file 11: 168 sessions
Finished file 12: 270 sessions
Finished file 13: 194 sessions
Finished file 14: 206 sessions
Finished file 15: 199 sessions
Finished file 16: 185 sessions
Finished file 17: 183 sessions
Finished file 18: 173 sessions
Finished file 19: 192 sessions
Finished file 20: 179 sessions
Finished file 21: 177 sessions
Finished file 22: 178 sessions
Finished file 23: 172 sessions
Finished file 24: 168 sessions


In [37]:
session_summary = (
    pd.concat(
        session_summary_parts,
        ignore_index=True
    )
    .groupby("session_id", as_index=False)
    .agg(
        rows=("rows", "sum"),
        positive_rows=("positive_rows", "sum")
    )
)

In [38]:
session_summary["has_positive"] = (
    session_summary["positive_rows"] > 0
)

In [39]:
print(
    "Total eligible sessions:",
    len(session_summary)
)

print(
    "Sessions with positive:",
    session_summary["has_positive"].sum()
)

print(
    "Sessions without positive:",
    (~session_summary["has_positive"]).sum()
)

print(
    "Total eligible rows:",
    session_summary["rows"].sum()
)

print(
    "Total positive rows:",
    session_summary["positive_rows"].sum()
)

display(
    session_summary[
        session_summary["has_positive"]
    ]
    .sort_values(
        "positive_rows",
        ascending=False
    )
    .head(10)
)

Total eligible sessions: 3457
Sessions with positive: 189
Sessions without positive: 3268
Total eligible rows: 10877668
Total positive rows: 2851


,session_id,rows,positive_rows,has_positive
2729,mEUjuFkRAf8WO9gFf33Gvdl4iC82ZsOOpZQYWyItw4M=,18879,506,True
3238,vf8sEl+uWrdO3Wzf2vwhBlEu1dL0IjFyLFeUH/9f3m0=,5626,242,True
1759,UKdcz6CZAs8fgWOq9+8olhGXa0iaMvdi+YqmHO6WdHw=,601,197,True
1197,K4YDiLzjIxHttrdpO7FFc/BVB7dob0bvTsEpP5snfNc=,5454,168,True
3020,ru4QoZ0yBpIFh5m/aDpx+fzkXkF2VBz4klZiZXqwwcQ=,222,155,True
1347,Ml6t5ueMh1+vJwSFBMiN1ZVHOUXpr4ICWrbS/W3JDAA=,189,118,True
2259,dmbOMTCzgPxfvM2sbM6NNVePFScKsPEFV9HiPYIgBqA=,210,81,True
911,F+e+VBB3zTQJLXm5QpbThltn0jYd11Fb/DtnpepFvMI=,96,77,True
1117,IkRg/7vL2GrwzBe3N7i16QNaPhYfTi9jUwgeynTiKj8=,116,56,True
758,CQ+hIMlnJcu9rsujWZm6S3yhfTuC3+9d907IyLBLaz4=,66,55,True


In [40]:
positive_sessions = (
    session_summary[
        session_summary["has_positive"]
    ]
    .sort_values(
        "positive_rows",
        ascending=False
    )
    .copy()
)

top_1_share = (
    positive_sessions["positive_rows"]
    .head(1)
    .sum()
    / positive_sessions["positive_rows"].sum()
)

top_5_share = (
    positive_sessions["positive_rows"]
    .head(5)
    .sum()
    / positive_sessions["positive_rows"].sum()
)

top_10_share = (
    positive_sessions["positive_rows"]
    .head(10)
    .sum()
    / positive_sessions["positive_rows"].sum()
)

print(f"Top 1 share: {top_1_share:.2%}")
print(f"Top 5 share: {top_5_share:.2%}")
print(f"Top 10 share: {top_10_share:.2%}")

Top 1 share: 17.75%
Top 5 share: 44.48%
Top 10 share: 58.05%


In [41]:
session_summary = session_summary.copy()

session_summary["split_stratum"] = (
    "no_positive"
)

positive_mask = (
    session_summary["has_positive"]
)

positive_rank = (
    session_summary.loc[
        positive_mask,
        "positive_rows"
    ]
    .rank(method="first")
)

positive_bins = pd.qcut(
    positive_rank,
    q=5,
    labels=False
)

session_summary.loc[
    positive_mask,
    "split_stratum"
] = (
    "positive_level_"
    + positive_bins.astype(str)
)

In [42]:
print(
    session_summary[
        "split_stratum"
    ].value_counts()
)

split_stratum
no_positive         3268
positive_level_1      38
positive_level_3      38
positive_level_0      38
positive_level_4      38
positive_level_2      37
Name: count, dtype: int64


In [43]:
from sklearn.model_selection import (
    train_test_split
)

train_sessions, test_sessions = (
    train_test_split(
        session_summary,
        test_size=0.20,
        random_state=42,
        stratify=session_summary[
            "split_stratum"
        ]
    )
)

In [44]:
def describe_split(name, table):
    print(f"\n{name}")
    print("Sessions:", len(table))
    print("Rows:", table["rows"].sum())
    print(
        "Positive sessions:",
        table["has_positive"].sum()
    )
    print(
        "Positive rows:",
        table["positive_rows"].sum()
    )

describe_split(
    "TRAIN",
    train_sessions
)

describe_split(
    "TEST",
    test_sessions
)


TRAIN
Sessions: 2765
Rows: 8648637
Positive sessions: 151
Positive rows: 2190

TEST
Sessions: 692
Rows: 2229031
Positive sessions: 38
Positive rows: 661


In [45]:
overlap = set(
    train_sessions["session_id"]
) & set(
    test_sessions["session_id"]
)

print(
    "Overlapping sessions:",
    len(overlap)
)

Overlapping sessions: 0


In [46]:
train_split = train_sessions[
    ["session_id"]
].copy()

train_split["split"] = "train"

test_split = test_sessions[
    ["session_id"]
].copy()

test_split["split"] = "test"

session_split = pd.concat(
    [train_split, test_split],
    ignore_index=True
)

split_path = (
    ROOT
    / "data"
    / "processed"
    / "session_split.parquet"
)

session_split.to_parquet(
    split_path,
    index=False
)

print("Saved to:", split_path)

Saved to: /Users/oliver/Downloads/video_rebuffering_risk_lab_fixed/data/processed/session_split.parquet


In [47]:
train_session_ids = set(
    train_sessions["session_id"]
)

In [48]:
tp = 0
fp = 0
fn = 0
tn = 0

In [49]:
for file_number, parquet_file in enumerate(
    parquet_files,
    start=1
):
    part = pd.read_parquet(
        parquet_file,
        columns=[
            "session_id",
            "eligible_prediction",
            "buffer",
            "target_5s"
        ]
    )

    train_part = part[
        part["eligible_prediction"]
        & part["session_id"].isin(
            train_session_ids
        )
    ]

    actual_positive = (
        train_part["target_5s"] == 1
    )

    predicted_positive = (
        train_part["buffer"] < 1.0
    )

    tp += int(
        (
            predicted_positive
            & actual_positive
        ).sum()
    )

    fp += int(
        (
            predicted_positive
            & ~actual_positive
        ).sum()
    )

    fn += int(
        (
            ~predicted_positive
            & actual_positive
        ).sum()
    )

    tn += int(
        (
            ~predicted_positive
            & ~actual_positive
        ).sum()
    )

    print(
        f"Finished file {file_number}"
    )

Finished file 1
Finished file 2
Finished file 3
Finished file 4
Finished file 5
Finished file 6
Finished file 7
Finished file 8
Finished file 9
Finished file 10
Finished file 11
Finished file 12
Finished file 13
Finished file 14
Finished file 15
Finished file 16
Finished file 17
Finished file 18
Finished file 19
Finished file 20
Finished file 21
Finished file 22
Finished file 23
Finished file 24


In [50]:
precision = tp / (tp + fp)
recall = tp / (tp + fn)
false_positive_rate = fp / (fp + tn)
alert_rate = (tp + fp) / (
    tp + fp + fn + tn
)

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)

print(f"Precision: {precision:.4%}")
print(f"Recall: {recall:.4%}")
print(
    f"False positive rate: "
    f"{false_positive_rate:.4%}"
)
print(f"Alert rate: {alert_rate:.4%}")

TP: 381
FP: 446
FN: 1809
TN: 8646001
Precision: 46.0701%
Recall: 17.3973%
False positive rate: 0.0052%
Alert rate: 0.0096%


In [51]:
thresholds = [
    0.5,
    1.0,
    2.0,
    3.0,
    5.0
]

results = {
    threshold: {
        "tp": 0,
        "fp": 0,
        "fn": 0,
        "tn": 0
    }
    for threshold in thresholds
}

In [52]:
for file_number, parquet_file in enumerate(
    parquet_files,
    start=1
):
    part = pd.read_parquet(
        parquet_file,
        columns=[
            "session_id",
            "eligible_prediction",
            "buffer",
            "target_5s"
        ]
    )

    train_part = part[
        part["eligible_prediction"]
        & part["session_id"].isin(
            train_session_ids
        )
    ]

    actual_positive = (
        train_part["target_5s"] == 1
    )

    for threshold in thresholds:
        predicted_positive = (
            train_part["buffer"]
            < threshold
        )

        results[threshold]["tp"] += int(
            (
                predicted_positive
                & actual_positive
            ).sum()
        )

        results[threshold]["fp"] += int(
            (
                predicted_positive
                & ~actual_positive
            ).sum()
        )

        results[threshold]["fn"] += int(
            (
                ~predicted_positive
                & actual_positive
            ).sum()
        )

        results[threshold]["tn"] += int(
            (
                ~predicted_positive
                & ~actual_positive
            ).sum()
        )

    print(f"Finished file {file_number}")

Finished file 1
Finished file 2
Finished file 3
Finished file 4
Finished file 5
Finished file 6
Finished file 7
Finished file 8
Finished file 9
Finished file 10
Finished file 11
Finished file 12
Finished file 13
Finished file 14
Finished file 15
Finished file 16
Finished file 17
Finished file 18
Finished file 19
Finished file 20
Finished file 21
Finished file 22
Finished file 23
Finished file 24


In [53]:
threshold_rows = []

for threshold in thresholds:
    counts = results[threshold]

    tp = counts["tp"]
    fp = counts["fp"]
    fn = counts["fn"]
    tn = counts["tn"]

    threshold_rows.append({
        "buffer_threshold": threshold,
        "alerts": tp + fp,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": tp / (tp + fp),
        "recall": tp / (tp + fn),
        "false_positive_rate": (
            fp / (fp + tn)
        ),
        "alert_rate": (
            (tp + fp)
            / (tp + fp + fn + tn)
        )
    })

threshold_results = pd.DataFrame(
    threshold_rows
)

display(
    threshold_results.style.format({
        "precision": "{:.2%}",
        "recall": "{:.2%}",
        "false_positive_rate": "{:.4%}",
        "alert_rate": "{:.4%}"
    })
)

,buffer_threshold,alerts,tp,fp,fn,precision,recall,false_positive_rate,alert_rate
0,0.500000,244,150,94,2040,61.48%,6.85%,0.0011%,0.0028%
1,1.000000,827,381,446,1809,46.07%,17.40%,0.0052%,0.0096%
2,2.000000,3973,891,3082,1299,22.43%,40.68%,0.0356%,0.0459%
3,3.000000,16318,1596,14722,594,9.78%,72.88%,0.1703%,0.1887%
4,5.000000,63746,2023,61723,167,3.17%,92.37%,0.7139%,0.7371%


In [54]:
import numpy as np
from sklearn.model_selection import (
    StratifiedKFold
)

development_sessions = (
    train_sessions
    .copy()
    .reset_index(drop=True)
)

development_sessions["cv_fold"] = -1

In [55]:
cv_splitter = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [56]:
for fold_number, (
    fitting_indices,
    validation_indices
) in enumerate(
    cv_splitter.split(
        X=np.zeros(
            len(development_sessions)
        ),
        y=development_sessions[
            "has_positive"
        ]
    )
):
    development_sessions.loc[
        validation_indices,
        "cv_fold"
    ] = fold_number

In [57]:
cv_summary = (
    development_sessions
    .groupby("cv_fold")
    .agg(
        sessions=("session_id", "size"),
        rows=("rows", "sum"),
        positive_sessions=(
            "has_positive",
            "sum"
        ),
        positive_rows=(
            "positive_rows",
            "sum"
        )
    )
)

display(cv_summary)

,sessions,rows,positive_sessions,positive_rows
cv_fold,,,,
0,553,1679097,30,146
1,553,1604284,30,321
2,553,1724956,30,887
3,553,1912232,30,569
4,553,1728068,31,267


In [58]:
print(
    "Unassigned sessions:",
    (
        development_sessions[
            "cv_fold"
        ] < 0
    ).sum()
)

print(
    "Total development sessions:",
    len(development_sessions)
)

print(
    "Unique development sessions:",
    development_sessions[
        "session_id"
    ].nunique()
)

Unassigned sessions: 0
Total development sessions: 2765
Unique development sessions: 2765


In [59]:
cv_path = (
    ROOT
    / "data"
    / "processed"
    / "development_cv_folds.parquet"
)

development_sessions[
    [
        "session_id",
        "has_positive",
        "positive_rows",
        "rows",
        "cv_fold"
    ]
].to_parquet(
    cv_path,
    index=False
)

print("Saved to:", cv_path)

Saved to: /Users/oliver/Downloads/video_rebuffering_risk_lab_fixed/data/processed/development_cv_folds.parquet


In [60]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import (
    StandardScaler
)

feature_columns = [
    "buffer",
    "size",
    "ssim_index",
    "cwnd",
    "in_flight",
    "min_rtt",
    "rtt",
    "delivery_rate",
    "cum_rebuf"
]

development_session_ids = set(
    development_sessions["session_id"]
)

scaler = StandardScaler()

missing_counts = pd.Series(
    0,
    index=feature_columns,
    dtype="int64"
)

infinite_counts = pd.Series(
    0,
    index=feature_columns,
    dtype="int64"
)

scaler_rows = 0

In [61]:
for file_number, parquet_file in enumerate(
    parquet_files,
    start=1
):
    part = pd.read_parquet(
        parquet_file,
        columns=[
            "session_id",
            "eligible_prediction",
            *feature_columns
        ]
    )

    development_part = part[
        part["eligible_prediction"]
        & part["session_id"].isin(
            development_session_ids
        )
    ]

    X_part = development_part[
        feature_columns
    ]

    missing_counts += (
        X_part.isna().sum()
    )

    numeric_values = X_part.to_numpy(
        dtype="float64"
    )

    infinite_counts += pd.Series(
        np.isinf(numeric_values).sum(axis=0),
        index=feature_columns
    )

    # 只有没有 missing/infinite 才交给 scaler
    valid_rows = np.isfinite(
        numeric_values
    ).all(axis=1)

    scaler.partial_fit(
        numeric_values[valid_rows]
    )

    scaler_rows += int(
        valid_rows.sum()
    )

    print(
        f"Finished file {file_number} | "
        f"valid development rows: "
        f"{valid_rows.sum():,}"
    )

Finished file 1 | valid development rows: 310,885
Finished file 2 | valid development rows: 331,618
Finished file 3 | valid development rows: 348,928
Finished file 4 | valid development rows: 337,434
Finished file 5 | valid development rows: 292,905
Finished file 6 | valid development rows: 363,917
Finished file 7 | valid development rows: 375,930
Finished file 8 | valid development rows: 364,983
Finished file 9 | valid development rows: 417,066
Finished file 10 | valid development rows: 283,637
Finished file 11 | valid development rows: 330,186
Finished file 12 | valid development rows: 433,912
Finished file 13 | valid development rows: 399,141
Finished file 14 | valid development rows: 402,904
Finished file 15 | valid development rows: 352,626
Finished file 16 | valid development rows: 358,012
Finished file 17 | valid development rows: 415,596
Finished file 18 | valid development rows: 374,053
Finished file 19 | valid development rows: 344,713
Finished file 20 | valid development row

In [62]:
scaler_summary = pd.DataFrame({
    "feature": feature_columns,
    "missing": missing_counts.values,
    "infinite": infinite_counts.values,
    "mean": scaler.mean_,
    "standard_deviation": scaler.scale_
})

print(
    "Rows used by scaler:",
    scaler_rows
)

display(scaler_summary)

Rows used by scaler: 8648637


,feature,missing,infinite,mean,standard_deviation
0,buffer,0,0,1.459518e+01,1.447821e+00
1,size,0,0,1.128337e+06,6.734762e+05
2,ssim_index,0,0,9.794331e-01,1.114876e-02
3,cwnd,0,0,1.059618e+03,7.079110e+02
4,in_flight,0,0,2.966671e-01,3.095589e+00
5,min_rtt,0,0,5.201769e+04,4.320386e+04
6,rtt,0,0,7.778108e+04,8.341877e+04
7,delivery_rate,0,0,9.911272e+06,1.166258e+07
8,cum_rebuf,0,0,2.194291e+00,1.559672e+01


In [63]:
from sklearn.linear_model import (
    SGDClassifier
)

logistic_model = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=0.0001,
    learning_rate="optimal",
    average=True,
    random_state=42
)

class_labels = np.array(
    [0, 1],
    dtype="int8"
)

first_training_batch = True
training_rows = 0
training_positives = 0

In [64]:
for file_number, parquet_file in enumerate(
    parquet_files,
    start=1
):
    part = pd.read_parquet(
        parquet_file,
        columns=[
            "session_id",
            "eligible_prediction",
            "target_5s",
            *feature_columns
        ]
    )

    fitting_part = part[
        part["eligible_prediction"]
        & part["session_id"].isin(
            development_session_ids
        )
    ]

    X_fitting = fitting_part[
        feature_columns
    ].to_numpy(
        dtype="float64"
    )

    y_fitting = fitting_part[
        "target_5s"
    ].to_numpy(
        dtype="int8"
    )

    X_fitting_scaled = scaler.transform(
        X_fitting
    )

    # 防止模型只按照原始时间顺序学习
    random_generator = (
        np.random.default_rng(
            42 + file_number
        )
    )

    row_order = (
        random_generator.permutation(
            len(y_fitting)
        )
    )

    X_fitting_scaled = (
        X_fitting_scaled[row_order]
    )

    y_fitting = y_fitting[row_order]

    if first_training_batch:
        logistic_model.partial_fit(
            X_fitting_scaled,
            y_fitting,
            classes=class_labels
        )

        first_training_batch = False
    else:
        logistic_model.partial_fit(
            X_fitting_scaled,
            y_fitting
        )

    training_rows += len(y_fitting)

    training_positives += int(
        y_fitting.sum()
    )

    print(
        f"Finished file {file_number} | "
        f"rows={len(y_fitting):,} | "
        f"positives={y_fitting.sum():,}"
    )

Finished file 1 | rows=310,885 | positives=561
Finished file 2 | rows=331,618 | positives=371
Finished file 3 | rows=348,928 | positives=33
Finished file 4 | rows=337,434 | positives=65
Finished file 5 | rows=292,905 | positives=3
Finished file 6 | rows=363,917 | positives=13
Finished file 7 | rows=375,930 | positives=17
Finished file 8 | rows=364,983 | positives=3
Finished file 9 | rows=417,066 | positives=6
Finished file 10 | rows=283,637 | positives=212
Finished file 11 | rows=330,186 | positives=23
Finished file 12 | rows=433,912 | positives=46
Finished file 13 | rows=399,141 | positives=301
Finished file 14 | rows=402,904 | positives=55
Finished file 15 | rows=352,626 | positives=40
Finished file 16 | rows=358,012 | positives=56
Finished file 17 | rows=415,596 | positives=83
Finished file 18 | rows=374,053 | positives=13
Finished file 19 | rows=344,713 | positives=7
Finished file 20 | rows=386,196 | positives=26
Finished file 21 | rows=404,007 | positives=40
Finished file 22 | row

In [65]:
coefficient_table = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": (
        logistic_model.coef_[0]
    )
})

coefficient_table[
    "absolute_coefficient"
] = (
    coefficient_table[
        "coefficient"
    ].abs()
)

coefficient_table = (
    coefficient_table
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
)

print(
    "Training rows:",
    training_rows
)

print(
    "Training positives:",
    training_positives
)

display(coefficient_table)

Training rows: 8648637
Training positives: 2190


,feature,coefficient,absolute_coefficient
8,cum_rebuf,0.479306,0.479306
0,buffer,-0.404970,0.404970
3,cwnd,-0.075893,0.075893
2,ssim_index,-0.067439,0.067439
4,in_flight,-0.051348,0.051348
5,min_rtt,-0.032900,0.032900
1,size,0.017293,0.017293
6,rtt,0.013629,0.013629
7,delivery_rate,0.006029,0.006029


In [66]:
import joblib

model_path = (
    ROOT
    / "data"
    / "processed"
    / "logistic_model_v1.joblib"
)

joblib.dump(
    {
        "scaler": scaler,
        "model": logistic_model,
        "features": feature_columns
    },
    model_path
)

print("Saved to:", model_path)

Saved to: /Users/oliver/Downloads/video_rebuffering_risk_lab_fixed/data/processed/logistic_model_v1.joblib


In [67]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

project_root = Path(
    "/Users/oliver/Downloads/video_rebuffering_risk_lab_fixed"
)

labelled_dir = (
    project_root
    / "data"
    / "processed"
    / "labelled_sent_5s"
)

cv_path = (
    project_root
    / "data"
    / "processed"
    / "development_cv_folds.parquet"
)

# 读取 development session 名单
dev_folds = pd.read_parquet(cv_path)

# 找到全部标记后的 Parquet 分块
labelled_files = sorted(labelled_dir.glob("*.parquet"))

print("CV columns:")
print(dev_folds.columns.tolist())

print("\nDevelopment sessions:")
print(dev_folds["session_id"].nunique())

print("\nNumber of labelled files:")
print(len(labelled_files))

print("\nLabelled-data columns:")
print(pq.ParquetFile(labelled_files[0]).schema.names)

CV columns:
['session_id', 'has_positive', 'positive_rows', 'rows', 'cv_fold']

Development sessions:
2765

Number of labelled files:
24

Labelled-data columns:
['sent_time', 'session_id', 'index', 'expt_id', 'channel', 'video_ts', 'format', 'size', 'ssim_index', 'cwnd', 'in_flight', 'min_rtt', 'rtt', 'delivery_rate', 'buffer', 'cum_rebuf', 'client_event_time', 'last_client_event', 'playback_state', 'eligible_prediction', 'freeze_start_time', 'seconds_until_freeze', 'target_5s']


In [68]:
import numpy as np
import gc

dev_session_set = set(dev_folds["session_id"])

delivery_rate_chunks = []
eligible_rows_found = 0

for file_number, file_path in enumerate(labelled_files, start=1):
    part = pd.read_parquet(
        file_path,
        columns=[
            "session_id",
            "eligible_prediction",
            "delivery_rate",
        ],
    )

    is_development = part["session_id"].isin(dev_session_set)
    is_eligible = part["eligible_prediction"].fillna(False)

    keep = is_development & is_eligible

    rates = part.loc[keep, "delivery_rate"].to_numpy(
        dtype="float64",
        copy=True,
    )

    rates = rates[np.isfinite(rates)]

    delivery_rate_chunks.append(rates)
    eligible_rows_found += len(rates)

    del part, rates
    gc.collect()

    if file_number % 4 == 0 or file_number == len(labelled_files):
        print(
            f"Processed {file_number}/{len(labelled_files)} files"
        )

dev_delivery_rates = np.concatenate(delivery_rate_chunks)

rate_cutoffs = np.quantile(
    dev_delivery_rates,
    [1 / 3, 2 / 3],
)

low_medium_cutoff = rate_cutoffs[0]
medium_high_cutoff = rate_cutoffs[1]

print("\nEligible development rows:")
print(eligible_rows_found)

print("\nDelivery-rate cutoffs in bytes/second:")
print("Low → Medium:", low_medium_cutoff)
print("Medium → High:", medium_high_cutoff)

print("\nEquivalent cutoffs in Mbps:")
print("Low → Medium:", low_medium_cutoff * 8 / 1_000_000)
print("Medium → High:", medium_high_cutoff * 8 / 1_000_000)

# 已经得到分界点，不再保留八百多万个数值
del dev_delivery_rates
del delivery_rate_chunks
gc.collect()

Processed 4/24 files
Processed 8/24 files
Processed 12/24 files
Processed 16/24 files
Processed 20/24 files
Processed 24/24 files

Eligible development rows:
8648637

Delivery-rate cutoffs in bytes/second:
Low → Medium: 4071094.3333333326
Medium → High: 11219115.0

Equivalent cutoffs in Mbps:
Low → Medium: 32.56875466666666
Medium → High: 89.75292


0

In [69]:
buffer_bins = [
    -np.inf,
    1,
    2,
    5,
    np.inf,
]

buffer_labels = [
    "buffer < 1",
    "1 <= buffer < 2",
    "2 <= buffer < 5",
    "buffer >= 5",
]

rate_bins = [
    -np.inf,
    low_medium_cutoff,
    medium_high_cutoff,
    np.inf,
]

rate_labels = [
    "low",
    "medium",
    "high",
]

partial_summaries = []

for file_number, file_path in enumerate(labelled_files, start=1):
    part = pd.read_parquet(
        file_path,
        columns=[
            "session_id",
            "eligible_prediction",
            "buffer",
            "delivery_rate",
            "target_5s",
        ],
    )

    keep = (
        part["session_id"].isin(dev_session_set)
        & part["eligible_prediction"].fillna(False)
    )

    part = part.loc[
        keep,
        [
            "buffer",
            "delivery_rate",
            "target_5s",
        ],
    ].copy()

    part["buffer_group"] = pd.cut(
        part["buffer"],
        bins=buffer_bins,
        labels=buffer_labels,
        right=False,
    )

    part["rate_group"] = pd.cut(
        part["delivery_rate"],
        bins=rate_bins,
        labels=rate_labels,
        right=False,
    )

    part_summary = (
        part.groupby(
            ["buffer_group", "rate_group"],
            observed=True,
        )
        .agg(
            rows=("target_5s", "size"),
            positive_rows=("target_5s", "sum"),
        )
        .reset_index()
    )

    partial_summaries.append(part_summary)

    del part, part_summary
    gc.collect()

    if file_number % 4 == 0 or file_number == len(labelled_files):
        print(
            f"Processed {file_number}/{len(labelled_files)} files"
        )

delivery_buffer_summary = (
    pd.concat(partial_summaries, ignore_index=True)
    .groupby(
        ["buffer_group", "rate_group"],
        observed=True,
    )[["rows", "positive_rows"]]
    .sum()
    .reset_index()
)

delivery_buffer_summary["positive_rate"] = (
    delivery_buffer_summary["positive_rows"]
    / delivery_buffer_summary["rows"]
)

delivery_buffer_summary["positives_per_10k_rows"] = (
    delivery_buffer_summary["positive_rate"] * 10_000
)

delivery_buffer_summary["buffer_group"] = pd.Categorical(
    delivery_buffer_summary["buffer_group"],
    categories=buffer_labels,
    ordered=True,
)

delivery_buffer_summary["rate_group"] = pd.Categorical(
    delivery_buffer_summary["rate_group"],
    categories=rate_labels,
    ordered=True,
)

delivery_buffer_summary = (
    delivery_buffer_summary
    .sort_values(["buffer_group", "rate_group"])
    .reset_index(drop=True)
)

print("\nRows represented:")
print(delivery_buffer_summary["rows"].sum())

print("\nPositive rows represented:")
print(delivery_buffer_summary["positive_rows"].sum())

display(
    delivery_buffer_summary.style.format(
        {
            "rows": "{:,}",
            "positive_rows": "{:,}",
            "positive_rate": "{:.6%}",
            "positives_per_10k_rows": "{:.3f}",
        }
    )
)

Processed 4/24 files
Processed 8/24 files
Processed 12/24 files
Processed 16/24 files
Processed 20/24 files
Processed 24/24 files

Rows represented:
8648637

Positive rows represented:
2190


,buffer_group,rate_group,rows,positive_rows,positive_rate,positives_per_10k_rows
0,buffer < 1,low,647,305,47.140649%,4714.065
1,buffer < 1,medium,177,76,42.937853%,4293.785
2,buffer < 1,high,3,0,0.000000%,0.000
3,1 <= buffer < 2,low,"2,553",475,18.605562%,1860.556
4,1 <= buffer < 2,medium,585,34,5.811966%,581.197
5,1 <= buffer < 2,high,8,1,12.500000%,1250.000
6,2 <= buffer < 5,low,"42,770","1,095",2.560206%,256.021
7,2 <= buffer < 5,medium,"11,237",31,0.275874%,27.587
8,2 <= buffer < 5,high,"5,766",6,0.104058%,10.406
9,buffer >= 5,low,"2,836,909",131,0.004618%,0.462


In [70]:
session_split_path = (
    project_root
    / "data"
    / "processed"
    / "session_split.parquet"
)

session_split = pd.read_parquet(session_split_path)

print("Columns:")
print(session_split.columns.tolist())

print("\nFirst five rows:")
display(session_split.head())

if "split" in session_split.columns:
    print("\nSession counts:")
    print(session_split["split"].value_counts())

    test_sessions = set(
        session_split.loc[
            session_split["split"] == "test",
            "session_id",
        ]
    )

    development_sessions = set(
        session_split.loc[
            session_split["split"] == "train",
            "session_id",
        ]
    )

    print("\nDevelopment sessions:")
    print(len(development_sessions))

    print("\nTest sessions:")
    print(len(test_sessions))

    print("\nOverlap:")
    print(len(development_sessions & test_sessions))

Columns:
['session_id', 'split']

First five rows:


,session_id,split
0,og+wDfEuudyJkgcVmMbE4T4xqc/RcHaD9Gs2rNv3o2o=,train
1,RCT6yoBFzcJpEBAkwVSw4wA2mLwwF22X9atXenKFBjs=,train
2,NCuUCUwCGGugDITfFodKZqvwKuymoHU0jFT6r9xwhM8=,train
3,mfVa38WxhK2qLHnOxd5jBpSAz8K/u/Od44uiL2tus7g=,train
4,oVAk2/eJBUOkLvEyjapLyU/Kzqxlfa2kPz2v0JHi904=,train



Session counts:
split
train    2765
test      692
Name: count, dtype: int64

Development sessions:
2765

Test sessions:
692

Overlap:
0


In [73]:
saved_artifact = joblib.load(model_path)

print("Saved object type:")
print(type(saved_artifact).__name__)

print("\nSaved keys:")
print(saved_artifact.keys())

Saved object type:
dict

Saved keys:
dict_keys(['scaler', 'model', 'features'])


In [75]:
saved_artifact = joblib.load(model_path)

scaler = saved_artifact["scaler"]
model = saved_artifact["model"]
feature_columns = saved_artifact["features"]

print("Model type:")
print(type(model).__name__)

print("\nScaler type:")
print(type(scaler).__name__)

print("\nFeatures:")
print(feature_columns)

print("\nModel classes:")
print(model.classes_)

print("\nCoefficient shape:")
print(model.coef_.shape)

Model type:
SGDClassifier

Scaler type:
StandardScaler

Features:
['buffer', 'size', 'ssim_index', 'cwnd', 'in_flight', 'min_rtt', 'rtt', 'delivery_rate', 'cum_rebuf']

Model classes:
[0 1]

Coefficient shape:
(1, 9)


In [76]:
import numpy as np
import pandas as pd
import gc

# 这三个列表从空开始，保存每个Parquet文件产生的测试结果
test_target_chunks = []
test_probability_chunks = []
test_buffer_chunks = []

for file_number, file_path in enumerate(labelled_files, start=1):
    # 只读取测试需要的列
    part = pd.read_parquet(
        file_path,
        columns=[
            "session_id",
            "eligible_prediction",
            "target_5s",
            *feature_columns,
        ],
    )

    # 只保留：
    # 1. 属于test的session
    # 2. 当前可以进行预测的playing行
    keep = (
        part["session_id"].isin(test_sessions)
        & part["eligible_prediction"].fillna(False)
    )

    test_part = part.loc[keep].copy()

    if len(test_part) > 0:
        # 取出9个feature，形成模型能读取的数字矩阵
        X_test_part = test_part[feature_columns].to_numpy(
            dtype="float64"
        )

        # 确认没有NaN或无限值
        if not np.isfinite(X_test_part).all():
            raise ValueError(
                f"Missing or infinite feature found in "
                f"{file_path.name}"
            )

        # 使用训练数据学到的均值和标准差进行转换
        X_test_scaled = scaler.transform(X_test_part)

        # 取target=1，也就是未来5秒freeze的预测概率
        probability_part = model.predict_proba(
            X_test_scaled
        )[:, 1]

        # 保存真实答案
        test_target_chunks.append(
            test_part["target_5s"].to_numpy(
                dtype="int8",
                copy=True,
            )
        )

        # 保存模型预测概率
        test_probability_chunks.append(
            probability_part.astype("float32")
        )

        # 保存buffer，之后与buffer baseline比较
        test_buffer_chunks.append(
            test_part["buffer"].to_numpy(
                dtype="float32",
                copy=True,
            )
        )

        del (
            test_part,
            X_test_part,
            X_test_scaled,
            probability_part,
        )

    del part
    gc.collect()

    if file_number % 4 == 0 or file_number == len(labelled_files):
        print(
            f"Processed {file_number}/{len(labelled_files)} files"
        )

# 把24个文件的结果拼成完整test结果
y_test = np.concatenate(test_target_chunks)
test_probability = np.concatenate(test_probability_chunks)
test_buffer = np.concatenate(test_buffer_chunks)

print("\nTest rows:")
print(len(y_test))

print("\nTest positives:")
print(y_test.sum())

print("\nTest prevalence:")
print(f"{y_test.mean():.6%}")

print("\nPredicted-probability summary:")
print(
    pd.Series(test_probability).describe(
        percentiles=[0.5, 0.9, 0.99, 0.999]
    )
)

Processed 4/24 files
Processed 8/24 files
Processed 12/24 files
Processed 16/24 files
Processed 20/24 files
Processed 24/24 files

Test rows:
2229031

Test positives:
661

Test prevalence:
0.029654%

Predicted-probability summary:
count    2.229031e+06
mean     4.173547e-04
std      8.576590e-03
min      3.490707e-05
50%      1.849073e-04
90%      2.210774e-04
99%      3.333652e-03
99.9%    1.900884e-02
max      9.999874e-01
dtype: float64


In [77]:
from sklearn.metrics import average_precision_score

# Logistic Regression的风险排序
model_ap = average_precision_score(
    y_test,
    test_probability,
)

# 随机排序的理论基准约等于positive发生率
random_baseline_ap = y_test.mean()

# 简单baseline：
# buffer越低风险越高，所以使用负buffer作为风险分数
buffer_only_ap = average_precision_score(
    y_test,
    -test_buffer,
)

print("Random baseline AP:")
print(f"{random_baseline_ap:.8f}")
print(f"{random_baseline_ap:.6%}")

print("\nBuffer-only AP:")
print(f"{buffer_only_ap:.8f}")
print(f"{buffer_only_ap:.6%}")

print("\nLogistic model AP:")
print(f"{model_ap:.8f}")
print(f"{model_ap:.6%}")

print("\nModel lift over random:")
print(f"{model_ap / random_baseline_ap:.2f}x")

print("\nModel lift over buffer-only:")
print(f"{model_ap / buffer_only_ap:.2f}x")

Random baseline AP:
0.00029654
0.029654%

Buffer-only AP:
0.36160168
36.160168%

Logistic model AP:
0.54771292
54.771292%

Model lift over random:
1847.00x

Model lift over buffer-only:
1.51x


## Phase 2 final research summary

### Final prediction problem

- One observation is an eligible `video_sent` row while playback state is `playing`.
- `target_5s = 1` when the same `session_id + index` has an official `event == "rebuffer"` strictly after `sent_time` and within five seconds.
- Startup, rows already rebuffering, and unknown playback state are excluded.
- Future ACK time, `freeze_start_time`, and `seconds_until_freeze` are not model features.

### Important correction

The early `cum_rebuf`-increment definition was rejected after a manual timeline showed that cumulative rebuffer time can increase at the later `play/resume` row. The authoritative freeze-start definition is `event == "rebuffer"`, yielding 5,935 freeze starts across 671 streams with no duplicate freeze keys.

### Processed dataset

```text
All sent rows:             11,900,686
Eligible playing rows:     10,877,668
Positive rows:                  2,851
Unknown-state rows:            955,982
Partitioned Parquet size:        0.36 GB
```

All natural negatives were retained. The model therefore sees the real rare-event prevalence.

### Leakage-safe validation

The outer split uses `session_id`, not random rows:

```text
Development: 2,765 sessions, 8,648,637 rows, 2,190 positives
Test:          692 sessions, 2,229,031 rows,   661 positives
Session overlap: 0
```

The five positive-volume strata used for the outer split are not CV folds. Five separate development CV folds were saved for future model/threshold work. V1 was fixed without CV tuning.

### Baseline and model

The fixed development rule `buffer < 1 second` was conservative: Precision 46.07%, Recall 17.40%, FPR 0.0052%, and alert rate 0.0096%.

Logistic V1 is an L2-regularized `SGDClassifier(loss="log_loss")` trained incrementally on nine standardized sent-time features. The strongest standardized associations were `cum_rebuf = +0.479` and `buffer = -0.405`. Coefficients are conditional associations, not causal effects or percentages.

A development-only diagnostic found that higher delivery rate generally corresponded to lower positive rate inside sufficiently populated buffer bands. Its fitted coefficient was only `+0.006`, so it should be interpreted as near zero and affected by correlation/additive-model limitations—not as evidence that faster delivery causes freezes.

### Final locked-test ranking result

| Risk score | Average Precision |
|---|---:|
| Random / prevalence | 0.0002965 |
| Buffer only (`-buffer`) | 0.3616017 |
| Logistic V1 | **0.5477129** |

Logistic V1 improved AP over buffer only by 0.1861 absolute (1.51x relative). This supports useful row-level risk ranking on unseen sessions.

### What this result does not mean

- AP 0.548 does not mean that 54.8% of freezes were detected.
- AP is not accuracy, Recall, or Precision at one threshold.
- No Logistic probability threshold was selected or tuned on test.
- The 661 positives are positive rows, not necessarily 661 independent freeze episodes.
- The current model is not yet a deployable automatic quality-reduction policy.

### Main limitations

- One day of data and only 38 positive test sessions;
- session dependence and positive concentration;
- stale last-recorded buffer/cumulative-rebuffer values;
- unknown-state exclusions and possible right censoring;
- no event-level recall, calibration conclusion, or operational threshold;
- no historical trends, nonlinear terms, or formal subgroup evaluation.

Full interpretation and reporting guidance are in `docs/Phase2_Modelling_Report_CN.md` and `docs/RESEARCH_REPORT_GUIDE_CN.md`.
